Read our preprocessed data

In [57]:
data = pd.read_csv("/nlsasfs/home/mpplab-cdac/mitsa/Saurav/PH/processed/jets_g.csv")

In [2]:
!ls processed

jets_g.csv  jets_q.csv	jets_t.csv  jets_undef.csv  jets_w.csv	jets_z.csv


Read Jets and make persistence images and create data for CNN training.

In [ ]:
import numpy as np
import pandas as pd
import os
from ripser import ripser
from persim import PersistenceImager
import matplotlib.pyplot as plt

# --- Define persistence image parameters ---
pimg = PersistenceImager(pixel_size=0.005)
pimg.pixels = (40, 40)          # resolution
pimg.birth_range = (0, 0.20)    # zoomed-in region
pimg.pers_range  = (0, 0.20)
pimg.kernel_params = {'sigma': 0.001}  # localized kernel

# Paths
# Paths
input_dir = "/nlsasfs/home/mpplab-cdac/mitsa/Saurav/PH/processed/"
datasets = ["jets_g.csv", "jets_q.csv", "jets_t.csv", "jets_w.csv", "jets_z.csv"]
labels = {"jets_g.csv": 0, "jets_q.csv": 1, "jets_t.csv": 2, "jets_w.csv": 3, "jets_z.csv": 4}

X, y = [], []

for fname in datasets:
    df = pd.read_csv(os.path.join(input_dir, fname))
    df = df.iloc[:, 1:]  # smaller subset for demo
    tag = fname.replace(".csv", "")

    for j in range(len(df)):
        jet = df.iloc[j]

        # Extract features
        ptrel = np.array([jet[f"ptrel_{i}"] for i in range(1, 101)])
        etarot = np.array([jet[f"etarot_{i}"] for i in range(1, 101)])
        phirot = np.array([jet[f"phirot_{i}"] for i in range(1, 101)])

        # Mask padded entries
        mask = ptrel > 0.001
        pts = np.vstack([etarot[mask], phirot[mask]]).T
        if len(pts) < 5:
            continue

        # --- Persistent Homology ---
        dgms = ripser(pts)['dgms']
        H0 = dgms[0][np.isfinite(dgms[0][:, 1])]
        H1 = dgms[1][np.isfinite(dgms[1][:, 1])]

        # --- Persistence Images ---
        jet_features = []
        for d, H in enumerate([H0, H1]):
            if len(H) > 0:
                img = pimg.transform(H)
                img = img / (img.max() + 1e-8)

                # Save visualization for first jet of each dataset
                if j == 0:
                    plt.imshow(img, cmap="inferno",
                               extent=[pimg.birth_range[0], pimg.birth_range[1],
                                       pimg.pers_range[0], pimg.pers_range[1]],
                               origin="lower", aspect="auto")
                    plt.colorbar()
                    plt.title(f"Persistence Image H{d} ({tag})")
                    plt.savefig(f"{tag}_H{d}_PI.png", dpi=200)
                    plt.close()

                jet_features.append(img.flatten())
            else:
                jet_features.append(np.zeros(pimg.pixels[0] * pimg.pixels[1]))

        # Concatenate H0+H1
        jet_features = np.concatenate(jet_features)
        X.append(jet_features)
        y.append(labels[fname])

# Convert to arrays
X = np.array(X)
y = np.array(y)

print("Feature matrix shape:", X.shape)
print("Labels:", np.unique(y, return_counts=True))

# Save arrays
np.save("X_persistence.npy", X)
np.save("y_labels.npy", y)
print("Saved X_persistence.npy and y_labels.npy")
